# Modèle v3 — apports des guides, testés sur les données

**Cellule 1** : simulation mathématique v3 (modèle final + marchés dérivés + fiabilité + comparaison optionnelle aux cotes).

**Cellule 2** : vérification chiffrée des méthodes présentes dans les 5 guides (à lancer après la cellule 1).

Cadre académique : les cotes servent de référence de comparaison, pas de support de pari.

In [1]:
# ==================== SIMULATION MATHÉMATIQUE — VERSION 3 (apports des guides testés) ====================
# Nouveautés v3, issues des guides et retenues APRÈS test sur les données :
#   a. niveau de buts récent de la ligue (étape 1 de la méthode Poisson « force d'attaque × défense ») → modèle de buts mieux calibré
#   b. marchés dérivés de la grille de scores : cage inviolée, les deux marquent, but en 1re mi-temps, over/under
#   c. conversion probabilité ↔ cote juste (1/p) et comparaison optionnelle avec les cotes d'un bookmaker (méthode de Shin)
#   d. indice de fiabilité : précision historique du modèle selon sa confiance + alerte fin de saison (moins prévisible)
# Améliorations testées et retenues (gain de log loss démontré sur 2021/22 → 2025/26) :
#   1. pi-ratings (force domicile / extérieur séparée, Constantinou & Fenton 2013)
#   2. « ouverture » attendue du match (xG approché pour + contre des deux équipes) → mieux estimer le nul
#   3. |ΔElo| (le nul est plus probable quand les forces sont proches) et effet propre à chaque ligue
#   4. pondération temporelle : un match d'il y a 4 saisons compte 2 fois moins qu'un match récent
#   5. réestimation chaque saison (walk-forward) pour la validation, et sur toutes les données pour prédire
import numpy as np, pandas as pd, difflib, warnings
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from scipy.stats import poisson
warnings.filterwarnings("ignore")

LIGUES = {"E0": "Premier League", "SP1": "Liga", "I1": "Serie A", "D1": "Bundesliga", "F1": "Ligue 1"}
DEMI_VIE = 4                                                    # saisons
URL = "https://raw.githubusercontent.com/xgabora/Club-Football-Match-Data-2000-2025/main/data/Matches.csv"
raw = pd.read_csv(URL, parse_dates=["MatchDate"], low_memory=False)
raw["Season"] = np.where(raw.MatchDate.dt.month >= 7, raw.MatchDate.dt.year, raw.MatchDate.dt.year - 1)

# ---------- 1. Pi-ratings sur TOUTES les ligues (les promus arrivent avec une note) ----------
R_all = raw.dropna(subset=["HomeTeam", "AwayTeam"]).sort_values("MatchDate", kind="stable").reset_index(drop=True)
PI = {}
def pi_ratings(d, lr=0.035, gamma=0.7, b=10, c=3):
    out = np.zeros((len(d), 4))
    for i, (h, a, gh, ga) in enumerate(zip(d.HomeTeam.values, d.AwayTeam.values, d.FTHome.values, d.FTAway.values)):
        rh = PI.setdefault(h, [0.0, 0.0]); ra = PI.setdefault(a, [0.0, 0.0])          # [note domicile, note extérieur]
        out[i] = rh[0], rh[1], ra[0], ra[1]                                          # valeurs AVANT le match
        if np.isnan(gh): continue
        eh = np.sign(rh[0]) * (b ** (abs(rh[0]) / c) - 1); ea = np.sign(ra[1]) * (b ** (abs(ra[1]) / c) - 1)
        e = (gh - ga) - (eh - ea); s = np.sign(e) * c * np.log10(1 + abs(e))
        o0, o1 = rh[0], ra[1]
        rh[0] += s * lr; rh[1] += (rh[0] - o0) * gamma; ra[1] -= s * lr; ra[0] += (ra[1] - o1) * gamma
    return out
R_all[["pi_hh", "pi_ha", "pi_ah", "pi_aa"]] = pi_ratings(R_all)

# ---------- 2. Variables d'avant-match, 5 grands championnats ----------
D = R_all[R_all.Division.isin(LIGUES) & (R_all.Season >= 2005)].dropna(subset=["FTResult"]).reset_index(drop=True)
D["mid"] = np.arange(len(D)); D["y"] = D.FTResult.map({"H": 0, "D": 1, "A": 2})
def side(home):
    s, o = ("Home", "Away") if home else ("Away", "Home")
    return pd.DataFrame({"mid": D.mid, "date": D.MatchDate, "Division": D.Division, "team": D[f"{s}Team"], "home": int(home),
                         "gf": D[f"FT{s}"], "ga": D[f"FT{o}"], "sot_f": D[f"{s}Target"], "sot_a": D[f"{o}Target"], "elo": D[f"{s}Elo"]})
L = pd.concat([side(True), side(False)]).sort_values(["team", "date", "mid"]).reset_index(drop=True)
c_L = L.groupby("Division").gf.sum() / L.groupby("Division").sot_f.sum()
L["xg_f"], L["xg_a"] = L.sot_f * L.Division.map(c_L), L.sot_a * L.Division.map(c_L)
g = L.groupby("team")
mf = g.xg_f.transform(lambda x: x.shift(1).rolling(10, min_periods=5).mean())
ma = g.xg_a.transform(lambda x: x.shift(1).rolling(10, min_periods=5).mean())
L["Q"], L["O"] = mf - ma, mf + ma                                      # qualité de jeu, ouverture
H, A = L[L.home == 1].set_index("mid"), L[L.home == 0].set_index("mid")
X = D.set_index("mid")
X["dElo"] = (X.HomeElo - X.AwayElo) / 100; X["absElo"] = X.dElo.abs()
X["dQ"] = H.Q - A.Q; X["Otot"] = H.O + A.O - 5.5                # centré sur la moyenne (~5,5)
X["pi_diff"] = X.pi_hh - X.pi_aa; X["pi_diff_avg"] = (X.pi_hh + X.pi_ha) / 2 - (X.pi_ah + X.pi_aa) / 2
for lg in ["SP1", "I1", "D1", "F1"]: X[f"lg_{lg}"] = (X.Division == lg).astype(float)   # Premier League = référence
FE = ["dElo", "absElo", "dQ", "Otot", "pi_diff", "pi_diff_avg", "lg_SP1", "lg_I1", "lg_D1", "lg_F1"]
Dm = D.sort_values(["Division", "MatchDate", "mid"])
for c, col in (("lg_mh", "FTHome"), ("lg_ma", "FTAway")):
    Dm[c] = Dm.groupby("Division")[col].transform(lambda x: x.shift(1).rolling(380, min_periods=150).mean())
X = X.join(Dm.set_index("mid")[["lg_mh", "lg_ma"]])
X["lg_mh_c"], X["lg_ma_c"] = X.lg_mh - 1.5, X.lg_ma - 1.15            # niveau de buts de la ligue (380 derniers matchs), centré
FE_B = FE + ["lg_mh_c", "lg_ma_c"]                                     # variables du modèle de buts
X = X.dropna(subset=FE_B)

def poids(TR, s_ref): return 0.5 ** ((s_ref - TR.Season) / DEMI_VIE)
def fit_logit(TR, s_ref): return LogisticRegression(C=1e6, max_iter=20000, tol=1e-8).fit(TR[FE], TR.y, sample_weight=poids(TR, s_ref))
def tau(x, y, lh, la, r):
    t = np.ones(np.broadcast(x, y, lh).shape)
    t = np.where((x == 0) & (y == 0), 1 - lh * la * r, t); t = np.where((x == 0) & (y == 1), 1 + lh * r, t)
    t = np.where((x == 1) & (y == 0), 1 + la * r, t);     t = np.where((x == 1) & (y == 1), 1 - r, t)
    return t
def fit_buts(TR, s_ref):
    w, Xa = poids(TR, s_ref), sm.add_constant(TR[FE_B])
    gh = sm.GLM(TR.FTHome, Xa, family=sm.families.Poisson(), freq_weights=w).fit()
    ga = sm.GLM(TR.FTAway, Xa, family=sm.families.Poisson(), freq_weights=w).fit()
    lh, la = gh.predict(Xa).values, ga.predict(Xa).values
    nll = lambda r: -np.sum(w * np.log(np.clip(tau(TR.FTHome.values, TR.FTAway.values, lh, la, r), 1e-12, None)))
    return gh, ga, minimize_scalar(nll, bounds=(-.3, .3), method="bounded").x
def grille(lh, la, rho, gmax=10):
    G = np.arange(gmax + 1)
    M = poisson.pmf(G[:, None], lh) * poisson.pmf(G[None, :], la) * tau(G[:, None], G[None, :], lh, la, rho)
    return M / M.sum()

# ---------- 3. Validation walk-forward (chaque saison prédite avec un modèle entraîné AVANT elle) ----------
te = X[X.Season >= 2021]; P_new = np.zeros((len(te), 3)); P_old = np.zeros((len(te), 3))
for s in sorted(te.Season.unique()):
    TR, m = X[X.Season < s], (te.Season == s).values
    P_new[m] = fit_logit(TR, s - 1).predict_proba(te.loc[m, FE])
    P_old[m] = LogisticRegression(C=1e6, max_iter=5000).fit(X.loc[X.Season <= 2020, ["dElo", "dQ"]], X.y[X.Season <= 2020]).predict_proba(te.loc[m, ["dElo", "dQ"]])
yt = te.y.values
llv = lambda P: -np.log(P[np.arange(len(yt)), yt])
d = llv(P_new) - llv(P_old); bs = d[np.random.default_rng(0).integers(0, len(d), (2000, len(d)))].mean(1)
inv = 1 / te[["OddHome", "OddDraw", "OddAway"]].values; Pm = inv / inv.sum(1, keepdims=True); ok = ~np.isnan(Pm).any(1)
print("=" * 80); print(f"VALIDATION WALK-FORWARD  (2021/22 → 2025/26 : {len(te)} matchs)  —  log loss, plus bas = mieux"); print("=" * 80)
print(f"  Modèle précédent (ΔElo + ΔQ)        {llv(P_old).mean():.4f}   bon résultat {np.mean(P_old.argmax(1) == yt):.1%}")
print(f"  Modèle amélioré                      {llv(P_new).mean():.4f}   bon résultat {np.mean(P_new.argmax(1) == yt):.1%}")
print(f"  Cotes des bookmakers (référence)     {-np.log(Pm[ok][np.arange(ok.sum()), yt[ok]]).mean():.4f}   bon résultat {np.mean(Pm[ok].argmax(1) == yt[ok]):.1%}")
gap_old = llv(P_old)[ok].mean() - (-np.log(Pm[ok][np.arange(ok.sum()), yt[ok]])).mean()
gap_new = llv(P_new)[ok].mean() - (-np.log(Pm[ok][np.arange(ok.sum()), yt[ok]])).mean()
print(f"  Gain : {1000 * d.mean():+.2f} ×10⁻³  (IC 95 % {1000 * np.percentile(bs, 2.5):+.2f} à {1000 * np.percentile(bs, 97.5):+.2f})"
      f"  → écart aux cotes réduit de {1 - gap_new / gap_old:.0%}")

# ---------- 4. Modèle final, estimé sur TOUTES les saisons (poids max sur la plus récente) ----------
S_MAX = int(X.Season.max())
LOGIT = fit_logit(X, S_MAX); GH, GA, RHO = fit_buts(X, S_MAX)
_lam = GH.predict(sm.add_constant(X[FE_B])).values + GA.predict(sm.add_constant(X[FE_B])).values; _m = X.HTHome.notna().values
_htg = (X.HTHome + X.HTAway).values[_m]
K_HT = minimize_scalar(lambda k: -np.sum(_htg * np.log(k * _lam[_m]) - k * _lam[_m]), bounds=(.2, .7), method="bounded").x   # part des buts en 1re MT
# fiabilité : précision observée du modèle (validation) selon sa confiance, et prévisibilité en fin de saison
CONF = pd.DataFrame({"pmax": P_new.max(1), "ok": P_new.argmax(1) == yt})
CONF_BINS = [0, .42, .50, .61, 1]; CONF_ACC = CONF.groupby(pd.cut(CONF.pmax, CONF_BINS), observed=True).ok.mean().values
N_EQ = D.groupby(["Division", "Season"]).HomeTeam.nunique()
cls = list(LOGIT.classes_)                                              # 0 = dom, 1 = nul, 2 = ext
coef = pd.DataFrame({"log[P(1)/P(X)]": np.r_[LOGIT.intercept_[0] - LOGIT.intercept_[1], LOGIT.coef_[0] - LOGIT.coef_[1]],
                     "log[P(2)/P(X)]": np.r_[LOGIT.intercept_[2] - LOGIT.intercept_[1], LOGIT.coef_[2] - LOGIT.coef_[1]]}, index=["constante"] + FE)
coef = coef.join(pd.DataFrame({"log λ_dom": GH.params.values, "log λ_ext": GA.params.values}, index=["constante"] + FE_B), how="outer").loc[["constante"] + FE_B]
print("\n" + "=" * 80); print(f"FORMULES DU MODÈLE FINAL  (estimées sur 2005/06 → {S_MAX}/{S_MAX + 1 - 2000:02d}, demi-vie {DEMI_VIE} saisons ; ρ = {RHO:+.3f})"); print("=" * 80)
print(coef.round(3).to_string(na_rep="—"))
print("Lecture : z1 = somme(coef × variable) de la 1re colonne, z2 de la 2e ; P(X) = 1/(1+e^z1+e^z2), P(1) = e^z1·P(X), P(2) = e^z2·P(X)")
print(f"But en 1re mi-temps : λ_MT = {K_HT:.3f} × (λ_dom + λ_ext)  (≈ {K_HT:.0%} des buts avant la pause)")

# ---------- 5. Simulation d'un match ----------
def etat(team):
    t = L[L.team == team].sort_values("date")
    if t.empty: raise ValueError(f"Équipe inconnue : « {team} ». Proches : {difflib.get_close_matches(team, L.team.unique(), 5)}")
    last = t.tail(10)
    lg = t.Division.iloc[-1]; lgm = D[D.Division == lg].tail(380); s_cur = int(D.Season.max())
    joues = int(((D.HomeTeam == team) | (D.AwayTeam == team))[D.Season == s_cur].sum())
    n_eq = N_EQ.get((lg, s_cur), 20)
    return {"elo": t.elo.dropna().iloc[-1], "Q": last.xg_f.mean() - last.xg_a.mean(), "O": last.xg_f.mean() + last.xg_a.mean(),
            "pi": PI.get(team, [0.0, 0.0]), "lg": lg, "date": t.date.iloc[-1].date(),
            "lg_mh": lgm.FTHome.mean(), "lg_ma": lgm.FTAway.mean(), "restants": 2 * (n_eq - 1) - joues}

def shin(odds):
    pi = 1 / np.asarray(odds, float); S = pi.sum()
    f = lambda z: (np.sqrt(z * z + 4 * (1 - z) * pi * pi / S) - z).sum() / (2 * (1 - z)) - 1
    z = minimize_scalar(lambda z: f(z) ** 2, bounds=(0, .4), method="bounded").x
    return (np.sqrt(z * z + 4 * (1 - z) * pi * pi / S) - z) / (2 * (1 - z)), S - 1

def simulation(dom, ext, elo_dom=None, elo_ext=None, cotes=None, n_sim=100_000, seed=0, detail=False):
    """cotes = (cote_1, cote_X, cote_2) optionnel : pour comparer le modèle au marché (analyse académique)."""
    a, b = etat(dom), etat(ext)
    ea, eb = elo_dom or a["elo"], elo_ext or b["elo"]
    v = {"dElo": (ea - eb) / 100, "dQ": a["Q"] - b["Q"], "Otot": a["O"] + b["O"] - 5.5,
         "pi_diff": a["pi"][0] - b["pi"][1], "pi_diff_avg": np.mean(a["pi"]) - np.mean(b["pi"])}
    v["absElo"] = abs(v["dElo"])
    for lg in ["SP1", "I1", "D1", "F1"]: v[f"lg_{lg}"] = float(a["lg"] == lg)
    v["lg_mh_c"], v["lg_ma_c"] = a["lg_mh"] - 1.5, a["lg_ma"] - 1.15
    xm = pd.DataFrame([v])[FE]; xb = pd.DataFrame([v])[FE_B]
    p = LOGIT.predict_proba(xm)[0]
    cl = coef.loc[["constante"] + FE]; z1 = cl["log[P(1)/P(X)]"].values @ np.r_[1, xm.values[0]]; z2 = cl["log[P(2)/P(X)]"].values @ np.r_[1, xm.values[0]]
    Xc = sm.add_constant(xb, has_constant="add"); lh, la = GH.predict(Xc).iloc[0], GA.predict(Xc).iloc[0]
    M = grille(lh, la, RHO)
    rng = np.random.default_rng(seed); idx = rng.choice(M.size, n_sim, p=M.ravel()); sx, sy = np.divmod(idx, M.shape[1]); tot = sx + sy
    # incertitude : bootstrap du modèle sur 20 rééchantillonnages (detail=False pour aller plus vite)
    if detail:
        boot = []
        for k in range(20):
            Xb = X.sample(len(X), replace=True, random_state=k); boot.append(fit_logit(Xb, S_MAX).predict_proba(xm)[0])
        lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    print("\n" + "=" * 80); print(f"SIMULATION MATHÉMATIQUE : {dom} – {ext}   ({LIGUES.get(a['lg'], a['lg'])})"); print("=" * 80)
    print(f"Elo {ea:.0f} vs {eb:.0f} (ΔElo {v['dElo']:+.2f})  ·  qualité de jeu {a['Q']:+.2f} vs {b['Q']:+.2f} (ΔQ {v['dQ']:+.2f})")
    print(f"pi-ratings dom {a['pi'][0]:+.2f} / ext {b['pi'][1]:+.2f}  ·  ouverture attendue {v['Otot'] + 5.5:.2f}  ·  données au {max(a['date'], b['date'])}")
    print(f"z1 = {z1:+.3f}   z2 = {z2:+.3f}")
    ic = (lambda k: f"   (IC 95 % {lo[k]:.1%} – {hi[k]:.1%})") if detail else (lambda k: "")
    print(f"\n  ► Victoire {dom:<18s} {p[0]:6.1%}{ic(0)}")
    print(f"  ► Match nul {'':17s} {p[1]:6.1%}{ic(1)}")
    print(f"  ► Victoire {ext:<18s} {p[2]:6.1%}{ic(2)}")
    cs_d, cs_e, btts = M[:, 0].sum(), M[0, :].sum(), M[1:, 1:].sum()
    ht = 1 - np.exp(-K_HT * (lh + la))
    print(f"\n  Buts attendus : {dom} {lh:.2f} · {ext} {la:.2f}   ·   Monte-Carlo ({n_sim:,}) : 1 {np.mean(sx > sy):.1%} · X {np.mean(sx == sy):.1%} · 2 {np.mean(sx < sy):.1%}")
    mk = [("Victoire " + dom, p[0]), ("Match nul", p[1]), ("Victoire " + ext, p[2]),
          ("Plus de 1,5 but", np.mean(tot > 1.5)), ("Plus de 2,5 buts", np.mean(tot > 2.5)), ("Plus de 3,5 buts", np.mean(tot > 3.5)),
          ("Les deux équipes marquent", btts), ("But en 1re mi-temps", ht),
          (f"Cage inviolée {dom}", cs_d), (f"Cage inviolée {ext}", cs_e)]
    print("\n  Marché                              Probabilité   Cote juste (1/p)")
    for nm, q in mk: print(f"  {nm:35s} {q:8.1%}      {1 / q:6.2f}")
    top = sorted(((M[i, j], f"{i}-{j}") for i in range(6) for j in range(6)), reverse=True)[:5]
    print("  Scores les plus probables : " + " · ".join(f"{s} {q:.1%}" for q, s in top))
    k = int(np.digitize(p.max(), CONF_BINS[1:-1]))
    niveau = ["faible", "moyenne", "bonne", "élevée"][k]
    print(f"\n  Fiabilité du pronostic : {niveau} — quand le modèle est aussi confiant, son issue favorite s'est réalisée {CONF_ACC[k]:.0%} du temps (2021–2026)")
    if min(a["restants"], b["restants"]) <= 5:
        print("  ⚠ Fin de saison : ces matchs sont historiquement moins prévisibles (skill score 6,9 % contre 8,9 %).")
    if cotes is not None:
        pm, marge = shin(cotes)
        print(f"\n  Comparaison aux cotes {tuple(cotes)} (marge {marge:.1%}, probabilités de Shin) :")
        for nm, qm, qc in zip(["1", "X", "2"], p, pm): print(f"    {nm} : modèle {qm:.1%} · marché {qc:.1%} · écart {100 * (qm - qc):+.1f} pts")
        print("    Rappel : le marché est en moyenne plus précis que ce modèle (log loss 0,972 contre 0,982) ; un écart est d'abord un signe d'information manquante.")
    return {"1": p[0], "X": p[1], "2": p[2], "buts_dom": lh, "buts_ext": la, "over25": np.mean(tot > 2.5),
            "btts": btts, "but_1re_MT": ht, "cs_dom": cs_d, "cs_ext": cs_e, "fiabilite": niveau}

simulation("Barcelona", "Real Madrid", detail=True)
simulation("Paris SG", "Marseille", cotes=(1.45, 4.60, 6.50))   # cotes fictives, pour illustrer la comparaison


VALIDATION WALK-FORWARD  (2021/22 → 2025/26 : 8918 matchs)  —  log loss, plus bas = mieux
  Modèle précédent (ΔElo + ΔQ)        0.9844   bon résultat 52.5%
  Modèle amélioré                      0.9819   bon résultat 52.8%
  Cotes des bookmakers (référence)     0.9716   bon résultat 53.7%
  Gain : -2.51 ×10⁻³  (IC 95 % -4.32 à -0.70)  → écart aux cotes réduit de 20%



FORMULES DU MODÈLE FINAL  (estimées sur 2005/06 → 2026/27, demi-vie 4 saisons ; ρ = -0.054)
             log[P(1)/P(X)]  log[P(2)/P(X)]  log λ_dom  log λ_ext
constante             0.437           0.103      0.436      0.210
dElo                  0.187          -0.271      0.116     -0.130
absElo                0.109           0.088      0.007      0.017
dQ                    0.271          -0.136      0.091     -0.099
Otot                  0.076           0.015      0.069      0.050
pi_diff               0.084          -0.464     -0.058     -0.244
pi_diff_avg           0.121           0.400      0.158      0.207
lg_SP1               -0.046          -0.143     -0.034     -0.057
lg_I1                -0.203          -0.091     -0.093     -0.031
lg_D1                -0.139          -0.043     -0.010     -0.020
lg_F1                -0.077          -0.021     -0.039     -0.012
lg_mh_c                   —               —      0.122      0.174
lg_ma_c                   —               —      


SIMULATION MATHÉMATIQUE : Barcelona – Real Madrid   (Liga)
Elo 1968 vs 1931 (ΔElo +0.37)  ·  qualité de jeu +0.96 vs +0.92 (ΔQ +0.03)
pi-ratings dom +1.74 / ext +1.37  ·  ouverture attendue 5.64  ·  données au 2026-08-31
z1 = +0.575   z2 = -0.199

  ► Victoire Barcelona           49.4%   (IC 95 % 47.6% – 51.9%)
  ► Match nul                    27.8%   (IC 95 % 26.0% – 29.2%)
  ► Victoire Real Madrid         22.8%   (IC 95 % 21.1% – 23.8%)

  Buts attendus : Barcelona 1.62 · Real Madrid 1.08   ·   Monte-Carlo (100,000) : 1 48.9% · X 26.1% · 2 25.0%

  Marché                              Probabilité   Cote juste (1/p)
  Victoire Barcelona                     49.4%        2.02
  Match nul                              27.8%        3.60
  Victoire Real Madrid                   22.8%        4.39
  Plus de 1,5 but                        75.7%        1.32
  Plus de 2,5 buts                       50.7%        1.97
  Plus de 3,5 buts                       28.5%        3.51
  Les deux équipes ma

{'1': np.float64(0.5731466540109762),
 'X': np.float64(0.24085925940877534),
 '2': np.float64(0.1859940865802485),
 'buts_dom': np.float64(1.897784714103677),
 'buts_ext': np.float64(1.0589080518511857),
 'over25': np.float64(0.56755),
 'btts': np.float64(0.5609295169314533),
 'but_1re_MT': np.float64(0.7235276342673078),
 'cs_dom': np.float64(0.3468343349337606),
 'cs_ext': np.float64(0.14990109079033132),
 'fiabilite': 'bonne'}

## Vérification des méthodes des guides

In [2]:
# ==================== VÉRIFICATION DES MÉTHODES DES GUIDES (à lancer APRÈS la cellule « Simulation mathématique v3 ») ====================
# Chaque affirmation des guides est traduite en hypothèse testable, puis mesurée sur 2005/06 → 2025/26 (5 grands championnats).
import numpy as np, pandas as pd, statsmodels.api as sm
from scipy.stats import poisson
R = []                                                              # tableau final des verdicts
def ajoute(source, idee, mesure, verdict): R.append({"guide": source, "affirmation": idee, "mesure sur les données": mesure, "verdict": verdict})
V = X[X.Season <= 2025].dropna(subset=["OddHome", "OddDraw", "OddAway", "Over25", "Under25"]).copy()
inv = 1 / V[["OddHome", "OddDraw", "OddAway"]].values; PM = inv / inv.sum(1, keepdims=True)
V["pOver"] = (1 / V.Over25) / (1 / V.Over25 + 1 / V.Under25)

# 1. « Surebet français » : jouer les 2 cotes les plus basses avec des mises 1/cote → « gagne 8 fois sur 10 »
o = V[["OddHome", "OddDraw", "OddAway"]].values; two = np.argsort(o, 1)[:, :2]
st = 1 / np.take_along_axis(o, two, 1); win = (two == V.y.values[:, None]).any(1); ret = np.where(win, 1, 0) / st.sum(1) - 1
ajoute("18 stratégies", "« Surebet français » gagne 8 fois sur 10", f"réussite {win.mean():.1%}, rendement {ret.mean():+.1%}", "Vrai en fréquence, perdant en moyenne (c'est une double chance)")
# 2. « 46 % de victoires à domicile → toute cote domicile > 2,17 est une value »
h = V[V.OddHome > 2.17]; rh = np.where(h.y == 0, h.OddHome - 1, -1).mean()
ajoute("18 stratégies", "Cote domicile > 2,17 = value (car 46 % de victoires dom.)", f"victoire dom. {np.mean(h.y == 0):.1%}, rendement {rh:+.1%}", "Faux : le taux moyen ne s'applique pas à un match précis")
# 3. Loi de Poisson « force d'attaque × potentiel de défense » (CEREPS)
lgp = D.groupby(["Division", "Season"]).agg(mh=("FTHome", "mean"), ma=("FTAway", "mean")).reset_index(); lgp["Season"] += 1
Lh, La = L[L.home == 1].copy(), L[L.home == 0].copy()
for T in (Lh, La):
    T["att"] = T.groupby("team").gf.transform(lambda x: x.shift(1).rolling(19, min_periods=8).mean())
    T["dfn"] = T.groupby("team").ga.transform(lambda x: x.shift(1).rolling(19, min_periods=8).mean())
W = V.reset_index().merge(lgp, on=["Division", "Season"]).set_index("mid").join(Lh.set_index("mid")[["att", "dfn"]].add_prefix("h_")).join(La.set_index("mid")[["att", "dfn"]].add_prefix("a_")).dropna(subset=["h_att", "a_att", "h_dfn", "a_dfn", "mh"])
lh, la = W.h_att * W.a_dfn / W.mh, W.a_att * W.h_dfn / W.ma
G = np.arange(11); PP = np.array([(lambda M: [np.tril(M, -1).sum(), np.trace(M), np.triu(M, 1).sum()])(poisson.pmf(G[:, None], a) * poisson.pmf(G[None, :], b)) for a, b in zip(lh, la)])
llc = -np.log(PP[np.arange(len(W)), W.y.values]).mean(); llv3 = -np.log(LOGIT.predict_proba(W[FE])[np.arange(len(W)), W.y.values]).mean()
ajoute("Kit foot (CEREPS)", "Poisson « attaque × défense » prédit le score", f"log loss 1X2 {llc:.3f} (modèle v3 : {llv3:.3f}, taux de base 1,075)", "Utile mais dépassé ; son étape 1 (niveau de buts de la ligue) a été intégrée au modèle de buts")
# 4. BTTS par ligue et « BTTS non » vs « moins de 2,5 buts »
bt = ((V.FTHome > 0) & (V.FTAway > 0)); b_lg = bt.groupby(V.Division).mean()
ajoute("Kit foot (CEREPS)", "BTTS : Bundesliga 56,7 %, Ligue 1 48,7 %", f"Bundesliga {b_lg['D1']:.1%}, Ligue 1 {b_lg['F1']:.1%}", "Confirmé (ordre de grandeur)")
ajoute("Kit foot (CEREPS)", "« BTTS non » rapporte 37 % de plus que « moins de 2,5 »", f"BTTS non {1 - bt.mean():.1%} vs moins de 2,5 {np.mean(V.FTHome + V.FTAway < 2.5):.1%}", "Faux : événements différents, conclusion tirée d'un seul match")
# 5. Heure du coup d'envoi en Premier League (BTTS plus faible à 15h)
pl = V[(V.Division == "E0")].dropna(subset=["MatchTime", "pOver"]).copy()
pl["h15"] = (pd.to_datetime(pl.MatchTime, format="%H:%M:%S", errors="coerce").dt.hour == 15).astype(float)
bts = ((pl.FTHome > 0) & (pl.FTAway > 0)).astype(float)
m5 = sm.Logit(bts, sm.add_constant(pd.DataFrame({"lo": np.log(pl.pOver / (1 - pl.pOver)), "h15": pl.h15}))).fit(disp=0)
ajoute("Kit foot (CEREPS)", "PL : BTTS 45,8 % à 15h contre 55-57 % aux autres horaires", f"15h {bts[pl.h15 == 1].mean():.1%} vs autres {bts[pl.h15 == 0].mean():.1%} ; à cotes égales p = {m5.pvalues['h15']:.2f}", "Non confirmé")
# 6. « But en 1re MT dans 70 % des matchs » et critère « fin de série » (deux 0-0 à la mi-temps de suite à domicile)
Hh = L[L.home == 1][["mid", "team", "date"]].merge(D[["mid", "HTHome", "HTAway"]], on="mid").dropna().sort_values(["team", "date"])
Hh["z"] = ((Hh.HTHome == 0) & (Hh.HTAway == 0)).astype(float); gz = Hh.groupby("team").z
Hh["serie"] = (gz.shift(1) == 1) & (gz.shift(2) == 1) & (gz.shift(3) == 0)
Hh = Hh.merge(V[["pOver"]].reset_index(), on="mid").dropna()
m6 = sm.Logit(1 - Hh.z, sm.add_constant(pd.DataFrame({"lo": np.log(Hh.pOver / (1 - Hh.pOver)), "serie": Hh.serie.astype(float)}))).fit(disp=0)
ajoute("Revenu 1000-5000 €", "70 % des matchs ont un but en 1re mi-temps", f"{1 - Hh.z.mean():.1%}", "Confirmé")
ajoute("Revenu 1000-5000 €", "Après deux 0-0 à la pause, un but en 1re MT devient plus probable (« fin de série »)",
       f"{1 - Hh[Hh.serie].z.mean():.1%} après la série vs {1 - Hh.z.mean():.1%} ; à cotes égales p = {m6.pvalues['serie']:.2f}", "Faux (sophisme du joueur)")
# 7. Filtre « éviter la fin de saison » (1XBET) : le modèle est-il moins fiable ?
V2 = V[V.Season >= 2021].copy(); Pv = LOGIT.predict_proba(V2[FE])        # (modèle final : indicatif, pas hors échantillon)
fin = (V2.MatchDate.dt.month.isin([5])).values
llm = -np.log(Pv[np.arange(len(V2)), V2.y.values]); prior = np.bincount(V.y, minlength=3) / len(V); llb = -np.log(prior[V2.y.values])
sk = lambda m: 1 - llm[m].mean() / llb[m].mean()
ajoute("1XBET Secret", "Écarter les matchs de fin de saison (moins prévisibles)", f"skill score mai {sk(fin):.1%} vs reste {sk(~fin):.1%}", "Plutôt confirmé : moins prévisible → signalé par l'indice de fiabilité")
# 8. « La masse surestime les grands clubs → parier contre les favoris »
fav = PM.argmax(1); pf = PM.max(1); oo = V[["OddHome", "OddDraw", "OddAway"]].values
r_fav = np.where(fav == V.y.values, oo[np.arange(len(V)), fav] - 1, -1)
dog = np.where(fav == 0, 2, 0); r_dog = np.where(dog == V.y.values, oo[np.arange(len(V)), dog] - 1, -1)
ajoute("1XBET Secret", "Stratégie « anti-favoris »", f"rendement favori {r_fav[fav != 1].mean():+.1%} vs outsider {r_dog[fav != 1].mean():+.1%}", "Faux : les outsiders sont les plus surcotés (biais favori-outsider)")
# 9. Kelly / mises en % de bankroll, « 99 % », « gain 88 % », etc. : non testables comme prédiction → hors modèle
ajoute("Tous", "Mise fixe 1-5 %, tenir un bilan, éviter les combinés", "cohérent avec les mathématiques du risque (voir analyse Kelly/combinés)", "Conseil de gestion, pas une amélioration de prédiction")

pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(R).set_index(["guide", "affirmation"]))


mesure sur les données  \
guide              affirmation                                                                                                                                                     
18 stratégies      « Surebet français » gagne 8 fois sur 10                                                                                      réussite 79.4%, rendement -4.3%   
                   Cote domicile > 2,17 = value (car 46 % de victoires dom.)                                                                victoire dom. 30.3%, rendement -6.4%   
Kit foot (CEREPS)  Poisson « attaque × défense » prédit le score                                                      log loss 1X2 1.013 (modèle v3 : 0.983, taux de base 1,075)   
                   BTTS : Bundesliga 56,7 %, Ligue 1 48,7 %                                                                                      Bundesliga 57.1%, Ligue 1 50.1%   
                   « BTTS non » rapporte 37 % de plus que « moins de 2,5 »                                                                  BTTS non 47.6% vs moins de 2,5 48.8%   
                   PL : BTTS 45,8 % à 15h contre 55-57 % aux autres horaires                                                 15h 55.0% vs autres 53.4% ; à cotes égales p = 0.32   
Revenu 1000-5000 € 70 % des matchs ont un but en 1re mi-temps                                                                                                              70.0%   
                   Après deux 0-0 à la pause, un but en 1re MT devient plus probable (« fin de série »)                  67.5% après la série vs 70.0% ; à cotes égales p = 0.33   
1XBET Secret       Écarter les matchs de fin de saison (moins prévisibles)                                                                    skill score mai 7.1% vs reste 8.9%   
                   Stratégie « anti-favoris »                                                                                          rendement favori -2.8% vs outsider -11.3%   
Tous               Mise fixe 1-5 %, tenir un bilan, éviter les combinés                                  cohérent avec les mathématiques du risque (voir analyse Kelly/combinés)   

                                                                                                                                                                                           verdict  
guide              affirmation                                                                                                                                                                      
18 stratégies      « Surebet français » gagne 8 fois sur 10                                                                        Vrai en fréquence, perdant en moyenne (c'est une double chance)  
                   Cote domicile > 2,17 = value (car 46 % de victoires dom.)                                                              Faux : le taux moyen ne s'applique pas à un match précis  
Kit foot (CEREPS)  Poisson « attaque × défense » prédit le score                                         Utile mais dépassé ; son étape 1 (niveau de buts de la ligue) a été intégrée au modèle...  
                   BTTS : Bundesliga 56,7 %, Ligue 1 48,7 %                                                                                                           Confirmé (ordre de grandeur)  
                   « BTTS non » rapporte 37 % de plus que « moins de 2,5 »                                                          Faux : événements différents, conclusion tirée d'un seul match  
                   PL : BTTS 45,8 % à 15h contre 55-57 % aux autres horaires                                                                                                          Non confirmé  
Revenu 1000-5000 € 70 % des matchs ont un but en 1re mi-temps                                                                                                                             Confirmé  
                   Après deux 0-0 à la p